# AQIS lab: phase 0 launcher (Colab)

Runs the lab on Colab exactly as on the cloud VM and Oracle. It does four things only: mount Google Drive, clone the private repository at the pinned tag `AQIS_TAG`, install the lab's locked environment, and call lab commands. Everything else lives in the repository.

**Before you start** (full steps: `docs/colab/COLAB_RUNBOOK.md`):
- The folder `MyDrive/AQIS_LAB` exists in your Google Drive.
- The Colab secret `GITHUB_TOKEN` (key icon on the left) holds a fine-grained, read-only token for this repository, with notebook access switched on.
- Edit only cell 1: `AQIS_TAG` and `RUN_ARGS`. Then run the cells from top to bottom. Run cell 5 before you close the session, so its compute units are logged.

In [ ]:
# 1) Settings: edit AQIS_TAG and RUN_ARGS only (docs/colab/COLAB_RUNBOOK.md). Then Google Drive is mounted.
AQIS_TAG = "aqis-lab-wp-0.5-colab-1"
REPO = "https://github.com/NasirAman-promcare/voice-bot-vad-harness.git"
WORKDIR = "/content/aqis"
DRIVE_MOUNT = "/content/drive"
AQIS_DRIVE = "/content/drive/MyDrive/AQIS_LAB"
RUN_OUT = "/content/drive/MyDrive/AQIS_LAB/results/wp05-smoke-clip-stats"
RUN_ARGS = ["--shard-clips", "10"]

from google.colab import drive

drive.mount(DRIVE_MOUNT)

In [ ]:
# 2) Clone the private repository at AQIS_TAG. The GITHUB_TOKEN secret is read inside this one git command's
#    environment; it is never stored in a variable, a URL, .git/config or any output.
import base64
import os
import subprocess

from google.colab import userdata

subprocess.run(["rm", "-rf", WORKDIR], check=True)
subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", AQIS_TAG, REPO, WORKDIR],
               env=dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                        GIT_CONFIG_VALUE_0="Authorization: Basic " + base64.b64encode(
                            ("x-access-token:" + userdata.get("GITHUB_TOKEN")).encode()).decode()),
               check=True)

In [ ]:
# 3) Install the lab's locked environment: .venv on the Colab probe's Python, hash-checked lock (a few minutes).
subprocess.run(["pip", "install", "--quiet", "uv==0.8.17"], check=True)
subprocess.run(["bash", "aqis_lab/deploy/dev_bootstrap.sh"], cwd=WORKDIR, check=True)

In [ ]:
# 4) Lab commands: Drive layout, unit log start, doctor (OC-0.5-01), smoke-pack runner (OC-0.5-02).
PY = WORKDIR + "/.venv/bin/python"
subprocess.run([PY, "-m", "aqis_lab", "drive", "layout", "--drive-mount", DRIVE_MOUNT], cwd=WORKDIR, check=True)
subprocess.run([PY, "-m", "aqis_lab", "unitlog", "start", "--drive-mount", DRIVE_MOUNT], cwd=WORKDIR, check=True)
subprocess.run([PY, "-m", "aqis_lab", "doctor", "--target", "colab", "--drive-mount", DRIVE_MOUNT,
                "--out", AQIS_DRIVE + "/logs"], cwd=WORKDIR, check=True)
subprocess.run([PY, "-m", "aqis_lab", "run", "--task", "clip-stats", "--pack", "data/packs/smoke/v1", "--out", RUN_OUT,
                "--seed", "0", "--drive-mount", DRIVE_MOUNT, *RUN_ARGS], cwd=WORKDIR, check=True)

In [ ]:
# 5) End of session: append this session's estimated compute units to AQIS_LAB/logs/units.csv.
subprocess.run([PY, "-m", "aqis_lab", "unitlog", "end", "--drive-mount", DRIVE_MOUNT], cwd=WORKDIR, check=True)